####  Bronze vs Silver Testing – Users Table

This document describes the data quality, integrity, and reconciliation tests performed for the `users` table during ingestion from Bronze to Silver.





####  Tables Under Test

- Bronze: `coffee.bronze.users`
- Silver: `coffee.silver.users`
- Quarantine: `coffee.silver.users_quarantine`

In [0]:

-- Test 1: Compare row counts between Bronze and Silver
-- Silver count should be <= Bronze count due to deduplication and filtering

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.users

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.users;

In [0]:
-- Test 2: Ensure mandatory columns are NOT NULL in Silver
-- user_id and registered_at must always be present

SELECT COUNT(*) AS invalid_silver_records
FROM coffee.silver.users
WHERE
  user_id IS NULL
  OR registered_at IS NULL;


In [0]:
-- Test 3: Ensure each user_id appears only once in Silver
-- Confirms deduplication logic is working correctly

SELECT user_id, COUNT(*) AS cnt
FROM coffee.silver.users
GROUP BY user_id
HAVING COUNT(*) > 1;


In [0]:
-- Test 4: Silver should not contain users that never existed in Bronze
-- Prevents phantom dimension records

SELECT user_id
FROM coffee.silver.users

EXCEPT

SELECT user_id
FROM coffee.bronze.users;


In [0]:
-- Test 5: All valid Bronze users should be present in Silver
-- Identifies valid users accidentally dropped

SELECT user_id
FROM coffee.bronze.users
WHERE
  user_id IS NOT NULL
  AND registered_at IS NOT NULL

EXCEPT

SELECT user_id
FROM coffee.silver.users;


In [0]:
-- Test 6: Validate gender values are standardized
-- Allowed values are 'male', 'female', or NULL

SELECT DISTINCT gender
FROM coffee.silver.users;


In [0]:
-- Test 7: Birthdate should not be in the future
-- Ensures logical consistency

SELECT COUNT(*) AS invalid_birthdates
FROM coffee.silver.users
WHERE birthdate > current_date();


In [0]:
describe table coffee.silver.menu_items